# 内存检测（sanitizer）

内存访问错误通常**不立即报错**——表现为结果错乱、偶发错误或挂死，人工排查成本极高。sanitizer 能检查其覆盖范围内的访问与同步记录，但不能证明所有执行路径均安全。


## 环境准备

本节代码面向 Ascend 950 环境。运行前先完成基础环境配置（详见 01.06）：设置设备号、关闭设备后端自动加载，并完成导入自检。

In [ ]:
import os

os.environ.setdefault('TILE_FWK_DEVICE_ID', '0')
os.environ['TORCH_DEVICE_BACKEND_AUTOLOAD'] = '0'

import pypto_pro.language as pl
import torch
import torch_npu

print('导入完成；设备编译、执行与精度仍需在 950 验证')

## 1. 检测范围

在 `@pl.jit()` 配置 `sanitizer=True` 开启，框架在四类调用点自动插桩，启动结束后统一检测：

<table style="margin-left:0; margin-right:auto; text-align:left;">
<tr><th style="text-align:left;">类别</th><th style="text-align:left;">检测内容</th></tr>
<tr><td>内存访问</td><td>GM 越界、Tile 越界、片上地址重叠</td></tr>
<tr><td>Mutex</td><td>mutex 配对错误</td></tr>
<tr><td>相位</td><td>AccPhase/STPhase 与 K 链结构错误</td></tr>
<tr><td>同步</td><td>sync_src/sync_dst 配对</td></tr>
</table>


## 2. 使用约束（五条）

1. **仅供调试**：插桩改变执行开销，性能测量前必须移除 `sanitizer=True`；
2. 每次启动在 GM 分配约 **1GB 日志缓冲**；同一位置重复缺陷聚合为一条并计数；
3. Ptr 只携带地址不携带大小——以指针为源的视图**不做**"声明 Shape 超出源"检查；
4. sync 配对检测**不跨核**：分处不同核的 sync_src/sync_dst 各自被判漏配对（误报），需人工甄别；
5. **SIMT 线程块内不检测**（GM/Tile/mutex/相位/sync 均不覆盖）——SIMT 的越界仍靠第 7 章的边界保护纪律。


## 3. 操作步骤

```python
@pl.jit(sanitizer=True, auto_mutex=True)
def kernel(x: pl.Tensor[[64, 64], pl.DT_FP16]):
    ...
```

- 默认 False，不开启时行为完全不变；
- 检出问题写入编译产物目录下 `pypto_sanitizer_report.txt`（同目录有原始记录；不同版本可能使用 `sanitizer_report.bin` 或 `sanitizer_report.<launch_id>.bin`），同一 Kernel 多次启动结果**汇总在同一文件**；当前实现只要有 finding（包括 `TILE_OVERLAP` 告警）就抛出 `SanitizerReplayError`，需结合报告判断是否真正存在冲突；
- 无问题则静默通过。


## 4. 何时用

外层 SIMD Kernel 的疑似越界/重叠可先开 sanitizer；若问题位于 SIMT 函数内部，应检查索引覆盖、调试 Tensor 和共享内存屏障。TILE_OVERLAP 是声明区间扫描的 WARNING，不能等同于必然的数据竞争，也不能承诺补同步后该告警自动消失。


### 同一个重叠提示，可能对应两种不同情况

若 A、B 同时活跃且地址区间交叉，后续写 B 可能破坏 A，属于资源规划错误。第 3 章统计量的 DN 与行主序视图则是有意共享同一份数据，两个视图按顺序解释同一个地址；需要说明布局、数据含义与生命周期，而不是仅凭 WARNING 就复制出第二份独立统计量。

阅读报告时记录涉及的 Tile、地址区间、操作和核号，再回到原始源码检查。sanitizer 的声明区间扫描可能持续报告有意别名，增加同步不一定使警告消失；SIMT 内部访问没有被覆盖时，“没有报告”也不能证明安全。


## 5. 课后练习

1. （判断）sanitizer 检出"Tile 内存区间重叠"但你的算法"恰好依赖先后覆写同一地址"——这算缺陷吗？
2. （填空）sanitizer 的 SIMT 覆盖情况是 ______；因此 SIMT 侧防越界的手段是 ______。

In [ ]:
!cat ./answer/08.03_answer.txt

## 6. 本节小结

- sanitizer 四类检测、五条约束（调试专用、1GB 缓冲、Ptr 盲区、sync 不跨核、SIMT 不检测）；
- 报告在编译产物目录，聚合追加；
- 与第 7 章联动时先分清外层 Kernel 与 SIMT 内部的检测边界。